<div align="center">

# Week 02: The Human I: Perception, Colour and Its Psychological Effects

**Human-Computer Interaction (BIL-429), hands-on Colab notebook**

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

[Week overview](https://github.com/utkukose/human-computer-interaction-NB-lecture/tree/main/weeks/week-02) · [Lecture page](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-02/lecture.html) · [Interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-02/lab.html) · [PDF notes](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-02/Week02_Lecture_Notes.pdf)

</div>

## How to use this Colab notebook

This Colab notebook is the hands-on part of the week. The concepts are explained on the [lecture page](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-02/lecture.html), and each section that applies a part of the lecture starts with a link to it. Save your own copy first with File, Save a copy in Drive, then run the cells in order with Shift+Enter. Exercises check your answer as soon as you run them. Each exercise has a reference solution in a collapsed cell: Try first, then open it. Running all cells at once also runs the reference solutions. Cells titled *Observe and reflect* ask open questions about what the code shows. They have no solution cell, and a short answer in a text cell of your copy (Insert, Text cell) is enough. Cells titled *Python note* explain a piece of the Python language used nearby. Nothing needs to be installed.

**Starting in the middle.** Each numbered section can also be opened on its own, for example from an orange Colab box of the lecture page. Its first code cell, *Catch-up*, runs the code of the earlier sections without their printouts and figures. When the Colab notebook runs from the top, the catch-up cells notice it and do nothing.

## 0. Setup

The first cell prepares the environment and defines the helper `check`, which compares an answer with the expected value and prints immediate feedback. It also defines the colour pairs, the texts, the status colours and the wayfinding lines of the clinic app, with the values of the lecture. The second cell keeps the expected values of the exercises in the table `ANSWERS`, with its code hidden, so that no answer stands next to its exercise.

In [ ]:
# Environment: Everything used by this course is preinstalled in Google Colab.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import io
import colorsys
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (7, 3.6),
                     "axes.grid": True, "grid.alpha": 0.3})

def check(name, value, expected=None, tol=1e-6):
    """Compares an answer with the expected value and prints immediate feedback.
    Without an expected value in the call, check takes the value and its tolerance
    from ANSWERS, the table of the next cell."""
    if expected is None:
        answer, tol = ANSWERS[name]
        expected = answer()
    if value is None:
        print(f"{name}: not answered yet. Replace None and run the cell again.")
        return False
    try:
        scale = max(1.0, abs(float(expected)))
        ok = abs(float(value) - float(expected)) <= tol * scale
    except (TypeError, ValueError):
        ok = value == expected
    print(f"{name}: " + ("correct." if ok else
                         "not yet. Compare your steps with the lecture."))
    return ok


# Colour pairs of the clinic app in Week 2: role, text colour, background
# colour, size class ("text" or "large") and whether WCAG exempts the pair.
CLINIC_PAIRS = [
    ("Body text",                          "#4A4A4A", "#FFFFFF", "text",  False),
    ("Dates and secondary text",           "#8C8C8C", "#FFFFFF", "text",  False),
    ("Placeholder in the fields",          "#A0A0A0", "#FFFFFF", "text",  False),
    ("Label of the main button",           "#FFFFFF", "#2F5D8A", "text",  False),
    ("Links",                              "#4A90E2", "#FFFFFF", "text",  False),
    ("Error message",                      "#E74C3C", "#FFFFFF", "text",  False),
    ("Label on the green status chip",     "#FFFFFF", "#2ECC71", "text",  False),
    ("Prompt on the kiosk",                "#FFE08A", "#0E2236", "text",  False),
    ("Header of the waiting room display", "#FFFFFF", "#C0392B", "large", False),
    ("Label of a disabled button",         "#BDBDBD", "#F2F2F2", "text",  True),
]
# Texts of the clinic app: role, x-height or height of the characters in mm,
# and the viewing distance in mm.
CLINIC_TEXTS = [
    ("Phone, body text",                          2.1,   350),
    ("Phone, captions under the appointments",    1.4,   350),
    ("Kiosk, prompts",                            2.0,   600),
    ("Display, queue codes from the last row",   40.0, 11000),
    ("Corridor signs, department names",         30.0,  6000),
]
# Status colours of the appointment list and the four wayfinding lines (Part C)
STATUS = {"confirmed": "#2ECC71", "cancelled": "#E74C3C"}
LINES = {"red": "#D0312D", "green": "#2E9E44", "blue": "#2D6CC0",
         "yellow": "#E8C21F"}
print("Setup complete. Running in Colab:", IN_COLAB)

In [ ]:
#@title Expected answers of the exercises (run this cell; open it only after trying the exercises)
ANSWERS = {
    "Exercise 2.1": (lambda: 0.625, 1e-6),
    "Exercise 2.2": (lambda: 133.5, 1e-6),
    "Exercise 2.3": (lambda: 2, 1e-6),
    "Exercise A": (lambda: 25 / (2 * np.tan(np.radians(0.2) / 2)) / 1000, 1e-3),
    "Exercise B": (lambda: 118, 1e-6),
    "Exercise C": (lambda: delta_e(simulate(hex_to_rgb("#2E8B57"), "deutan"), simulate(hex_to_rgb("#C98510"), "deutan")), 1e-3),
    "Exercise D": (lambda: -0.31 * 0.5 + 0.60 * 1.2, 1e-6),
    "Exercise E": (lambda: 5, 1e-6),
}

## Python step 2: Arrays and functions for colour and contrast

A screen mixes red, green and blue light, so every colour on it is three numbers, and a palette is a table of such numbers. Text sizes, distances and colours come in large sets, which the library NumPy stores as *arrays* &#91;[3](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-02/lecture.html#ref-3)&#93;. This step computes with whole arrays at once, selects values with conditions, writes functions with default values and turns colour codes into numbers. It ends with the operator `@`, which transforms many colours with one matrix &#91;[2](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-02/lecture.html#ref-2)&#93;.

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "CLINIC_PAIRS" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-02/"
               "NB02_perception_and_colour.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "python-step":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

### Arrays: Many numbers at once

In [ ]:
# Python practice: a small example for learning the language
sizes_pt = np.array([9, 10, 11, 12, 14, 16, 18, 24])   # font sizes in the app
sizes_mm = sizes_pt * 0.3528          # 1 point = 1/72 inch = 0.3528 mm
print(sizes_mm.round(2))
print(f"mean {sizes_mm.mean():.2f} mm | largest {sizes_mm.max():.2f} mm")
print(f"the largest size is {sizes_pt.max() / sizes_pt.min():.1f} times the smallest")

`np.array` turns a list into an array. Arithmetic with an array works on every element at once, so `sizes_pt * 0.3528` converts all eight sizes without a loop. This is called *vectorised* arithmetic. Arrays have methods such as `mean`, `max`, `min` and `round`. A list of a million values would need a loop in plain Python, while the array does the same work in one line and much faster.

### Masks: Selecting values with a condition

In [ ]:
# Python practice: a small example for learning the language
small = sizes_pt < 12                 # True where a size is below 12 pt
print(small)
print("sizes below 12 pt:", sizes_pt[small])
print(f"how many: {small.sum()} | share: {small.mean():.3f}")
print("the small sizes enlarged by a quarter:", sizes_pt[small] * 1.25)

A comparison with an array gives an array of `True` and `False`, called a *mask*. Indexing with a mask, as in `sizes_pt[small]`, keeps the elements where the mask is `True`. Because `True` counts as 1 and `False` as 0, `sum` counts the matches and `mean` gives their share. An audit of an app asks exactly such questions: How many texts are too small, and which ones?

### Colours as numbers

In [ ]:
# Python practice: a small example for learning the language
def hex_to_rgb(code):
    """'#2F5D8A' -> array([ 47,  93, 138]): red, green and blue from 0 to 255."""
    code = code.lstrip("#")
    return np.array([int(code[i:i + 2], 16) for i in (0, 2, 4)])

def rgb_to_hex(rgb):
    """The inverse of hex_to_rgb, with each channel rounded to a whole number."""
    r, g, b = np.clip(np.round(rgb), 0, 255).astype(int)
    return f"#{r:02X}{g:02X}{b:02X}"

blue = hex_to_rgb("#2F5D8A")          # the colour of the main button
print(blue, "->", rgb_to_hex(blue))
print("each channel as a share of 255:", (blue / 255).round(3))

A colour code such as `#2F5D8A` writes each channel as two hexadecimal digits, from `00` (0) to `FF` (255). The call `int("2F", 16)` reads a text as a number in base 16, which gives 47. The list comprehension `[... for i in (0, 2, 4)]` reads the three pairs. In the other direction, the format `:02X` writes a number as two hexadecimal digits in capitals. The function `np.clip` keeps the values between 0 and 255, and `astype(int)` turns them into whole numbers.

### Functions with default values

In [ ]:
# Python practice: a small example for learning the language
def mix(colour, other, w=0.5):
    """Mixes two colours: w = 0 gives colour, w = 1 gives other."""
    return (1 - w) * colour + w * other

white = np.array([255, 255, 255])
for w in [0.2, 0.4, 0.6, 0.8]:
    tint = mix(blue, white, w)
    print(f"w = {w}: {rgb_to_hex(tint)} {tint.round(1)}")
print("halfway to white with the default w:", rgb_to_hex(mix(blue, white)))

A parameter with a default value, such as `w=0.5`, may be left out of a call, and the function then uses the default. Mixing a colour with white gives a *tint*, the lighter shade that apps use for backgrounds and selected rows. The arithmetic works on all three channels at once, because `colour` and `other` are arrays.

### Two-dimensional arrays: A palette

In [ ]:
# Python practice: a small example for learning the language
names = ["ink", "button blue", "link blue", "error red", "chip green", "pale yellow"]
palette = np.array([hex_to_rgb(c) for c in
                    ["#1E2433", "#2F5D8A", "#4A90E2", "#E74C3C", "#2ECC71", "#FFE08A"]])
print("shape:", palette.shape)                 # 6 colours, 3 channels each
channel_mean = palette.mean(axis=1)            # one number per colour
for i in np.argsort(channel_mean):             # from the darkest to the lightest
    print(f"{names[i]:<12} {palette[i]} mean of the channels {channel_mean[i]:6.1f}")

A two-dimensional array is a table. The array `palette` has one row per colour and one column per channel, and `shape` gives its size as (rows, columns). The method `mean(axis=1)` averages along each row, so it gives one number per colour, while `axis=0` would average each column. The function `np.argsort` returns the positions that sort the values. The mean of the channels is only a rough measure of lightness. The eye is far more sensitive to green light than to blue, and Section 2 replaces this mean with the weighted measure of the lecture.

### A matrix times a colour

In [ ]:
# Python practice: a small example for learning the language
grey = np.full((3, 3), 1 / 3)       # every row takes the mean of the channels
swap = np.array([[0, 1, 0],         # new red   = old green
                 [1, 0, 0],         # new green = old red
                 [0, 0, 1]])        # new blue  = old blue
print("button blue in grey:   ", (grey @ blue).round(1))
print("red and green swapped: ", swap @ blue)
print("the whole palette swapped at once:")
print(palette @ swap.T)

The operator `@` multiplies a matrix with a vector: Each row of the matrix is multiplied element by element with the colour and summed, which gives one new channel. A palette stores the colours as rows, so `palette @ swap.T` transforms all of them at once, where `.T` turns the matrix around (transposes it). Section 3 uses exactly this operation with matrices from vision science, which show how colours look to people with a colour vision deficiency.

**Quick check.** What does `mix(blue, white, 1)` return, and why?

<details><summary>Answer</summary>

`array([255., 255., 255.])`, which is white. With `w = 1` the weight of `colour` is `1 - w = 0`, so only `other` remains.

</details>

### Exercises for Python step 2

**Exercise 2.1.** Store the share of the font sizes in `sizes_pt` that are 12 pt or larger in `share_large`. Use a mask instead of counting by hand.

In [ ]:
share_large = None   # your answer

check("Exercise 2.1", share_large)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
share_large = (sizes_pt >= 12).mean()

check("Exercise 2.1", share_large)

**Exercise 2.2.** Mix `blue` with `white` at `w = 0.25` and store the green channel of the result in `g_tint`. The channels of an array are numbered from 0, so green is element 1.

In [ ]:
g_tint = None   # your answer

check("Exercise 2.2", g_tint)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
g_tint = mix(blue, white, 0.25)[1]

check("Exercise 2.2", g_tint)

**Exercise 2.3.** Store the number of colours in `palette` whose mean of the channels is above 128 in `n_light`.

In [ ]:
n_light = None   # your answer

check("Exercise 2.3", n_light)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
n_light = int((palette.mean(axis=1) > 128).sum())

check("Exercise 2.3", n_light)

## 1. Visual angle: The texts of the clinic app and your own reading &#91;[5](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-02/lecture.html#ref-5)&#93;

*Lecture: [The eye and the visual angle](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-02/lecture.html#the-eye-and-the-visual-angle)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "swap" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-02/"
               "NB02_perception_and_colour.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-1":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Whether a text can be read depends on its size and its distance together. One function of two arrays answers the question for every text of an app at once. The same function then turns the codes that you read in Part A of the interactive lab into a threshold in degrees. A fit over 21 codes per student is work for a computer, not for a calculator.

**From the formulas to the code.** In the code, the height $h$ is `h`, the distance $d$ is `d` and the visual angle $\theta$ is returned in degrees. `np.arctan` gives the angle in radians, and `np.degrees` multiplies by $180/\pi = 57.30$. The inverse formula $h = 2 d \tan(\theta / 2)$ is the function `height_for_angle`.

In [ ]:
def visual_angle(h, d):
    """Visual angle in degrees of a height h seen from the distance d.
    h and d need the same unit, and both may be NumPy arrays."""
    return np.degrees(2 * np.arctan(h / (2 * d)))

def height_for_angle(theta_deg, d):
    """The height that subtends theta_deg degrees at the distance d."""
    return 2 * d * np.tan(np.radians(theta_deg) / 2)

theta = visual_angle(2.0, 600)                # the kiosk prompt of the lecture
print(f"kiosk prompt: {theta:.3f} deg = {60 * theta:.1f} arcmin")
print(f"small-angle rule 57.3 h / d: {57.3 * 2.0 / 600:.3f} deg")
print(f"height for 0.3 deg at 600 mm: {height_for_angle(0.3, 600):.2f} mm")

**Python note.** `np.arctan`, `np.tan`, `np.degrees` and `np.radians` work on single numbers and on whole arrays, so the two functions accept a single text or a column of a table.

In [ ]:
CRITICAL_DEG = 0.2     # critical print size for fluent reading, in degrees
texts = pd.DataFrame(CLINIC_TEXTS, columns=["text", "h_mm", "d_mm"])
texts["angle_deg"] = visual_angle(texts["h_mm"], texts["d_mm"])
texts["arcmin"] = 60 * texts["angle_deg"]
texts["fluent"] = texts["angle_deg"] >= CRITICAL_DEG
texts["h_for_0.3_mm"] = height_for_angle(0.3, texts["d_mm"])
print(texts.round(3).to_string(index=False))

**Python note.** A column of a pandas table behaves like a NumPy array, so `visual_angle(texts["h_mm"], texts["d_mm"])` computes all five angles at once. The comparison `>= CRITICAL_DEG` gives a mask with one value per text, and `to_string(index=False)` prints the table without its row numbers.

**Exercise A.** A pharmacy sign uses letters with an x-height of 25 mm. Up to which distance, in metres, does it reach the critical print size of 0.2 degrees? Solve $\theta = 2 \arctan(h / 2d)$ for $d$ and store the distance in metres in `d_max_m`.

In [ ]:
d_max_m = None   # your answer

check("Exercise A", d_max_m)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
d_max_m = 25 / (2 * np.tan(np.radians(0.2) / 2)) / 1000

check("Exercise A", d_max_m)

Part A of the [interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-02/lab.html) ends with the button *Copy the results for Colab*. Paste the copied text between the triple quotes of `MY_READING` and run the cells. The code always adds a simulated class of 24 students for comparison. Their thresholds are assumptions of this Colab notebook, not measurements, and your own codes appear as student 0.

In [ ]:
# Paste the results of lab Part A between the triple quotes.
MY_READING = """
"""

LEVELS = np.array([3, 4.5, 6, 8, 10.5, 14, 18])    # visual angles in arcmin

def simulated_reading_class(n_students=24, seed=RANDOM_STATE):
    """Three codes per level and student, with assumed thresholds and slopes."""
    rng = np.random.default_rng(seed)
    rows = []
    for student in range(1, n_students + 1):
        a50 = 6.5 * rng.lognormal(0, 0.25)       # threshold in arcmin
        s = rng.uniform(0.6, 1.4)                # how steeply the curve rises
        for angle in np.repeat(LEVELS, 3):
            p = 1 / (1 + np.exp(-(angle - a50) / s))
            rows.append([student, angle, int(rng.random() < p)])
    return pd.DataFrame(rows, columns=["student", "eye_arcmin", "correct"])

reading = simulated_reading_class()
if MY_READING.strip():
    mine = pd.read_csv(io.StringIO(MY_READING.strip()))
    mine["student"] = 0                          # student 0 is you
    reading = pd.concat([mine[["student", "eye_arcmin", "correct"]], reading])
    source = "your own codes from the interactive lab and a simulated class"
else:
    source = "a simulated class of 24 students"
print("Data:", source, "|", len(reading), "codes")
level = reading["eye_arcmin"].round(1)
print(reading.groupby(level)["correct"].agg(["count", "mean"]).round(2))

**Python note.** `np.repeat(LEVELS, 3)` repeats every level three times. `pd.concat` stacks two tables with the same columns, and `agg(["count", "mean"])` computes two summaries of every group at once.

In [ ]:
def fit_threshold(angle, correct):
    """Maximum-likelihood fit of p = 1 / (1 + exp(-(angle - a50) / s)) on a grid.
    Returns the threshold a50 in arcmin and the slope s."""
    a_grid = np.linspace(1, 30, 291)             # candidate thresholds
    s_grid = np.linspace(0.3, 6, 58)             # candidate slopes
    A, S = np.meshgrid(a_grid, s_grid, indexing="ij")
    x = np.asarray(angle, float)
    y = np.asarray(correct, float)
    p = 1 / (1 + np.exp(-(x - A[..., None]) / S[..., None]))
    p = np.clip(p, 1e-6, 1 - 1e-6)
    loglik = (y * np.log(p) + (1 - y) * np.log(1 - p)).sum(axis=-1)
    i, j = np.unravel_index(np.argmax(loglik), loglik.shape)
    return a_grid[i], s_grid[j]

rows = []
for student, d in reading.groupby("student"):
    a50, s = fit_threshold(d["eye_arcmin"], d["correct"])
    rows.append([student, a50, s, a50 + s * np.log(3)])
fits = pd.DataFrame(rows, columns=["student", "a50", "s", "a75"])
fits = fits.set_index("student")
print(fits.describe().loc[["mean", "50%", "min", "max"]].round(2))
if 0 in fits.index:
    print("your threshold a75:", round(fits.loc[0, "a75"], 2), "arcmin")

**Python note.** The grid holds 291 candidate thresholds and 58 slopes. `A[..., None]` adds a third axis, so that one line computes the probability of every code for all 16,878 combinations at once. This is *broadcasting*. The sum of the logarithms of the probabilities of what really happened is the log-likelihood, and `np.argmax` finds the combination that makes the observed answers most likely. `np.unravel_index` turns that position back into a row and a column of the grid.

In [ ]:
# Figure: proportion correct, the fitted curves and the thresholds of the class
x = np.linspace(1, 22, 200)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for student, f in fits.iterrows():
    you = student == 0
    ax[0].plot(x, 1 / (1 + np.exp(-(x - f["a50"]) / f["s"])),
               color="#5A4BB5" if you else "#C9C3E6", lw=2.5 if you else 1,
               zorder=3 if you else 1, label="you" if you else None)
share = reading.groupby(level)["correct"].mean()
ax[0].scatter(share.index, share.values, color="#C98510", zorder=4,
              label="share correct, all students")
ax[0].set_xlabel("visual angle of the code at the eye (arcmin)")
ax[0].set_ylabel("probability of reading it")
ax[0].set_title("Reading curves")
ax[0].legend(loc="lower right")
p95 = np.percentile(fits["a75"], 95)
ax[1].hist(fits["a75"], bins=10, color="#C9C3E6", edgecolor="white")
ax[1].axvline(p95, color="#B53F2E", ls="--", label=f"95th percentile {p95:.1f}")
if 0 in fits.index:
    ax[1].axvline(fits.loc[0, "a75"], color="#5A4BB5", lw=2.5, label="you")
ax[1].set_xlabel("threshold a75 (arcmin)")
ax[1].set_ylabel("students")
ax[1].set_title("Thresholds of the class")
ax[1].legend()
plt.tight_layout()
plt.show()
h_last = height_for_angle(p95 / 60, 11000)
print(f"95 percent of the class read a code of {p95:.1f} arcmin or less.")
print(f"At 11 m, the last row, a code needs at least {h_last:.0f} mm for them.")

**Python note.** `iterrows()` gives the rows of a table to a loop, one pair of index and row at a time. `np.percentile(values, 95)` returns the value below which 95 percent of the values lie. The threshold $a_{75}$ is the angle at which the fitted curve reaches 0.75, so $a_{75} = a_{50} + s \ln 3$.

**Observe and reflect.** Your threshold mixes your eyes, your screen and your distance. On a screen with large pixels, the pixels may set the limit before your eyes do. The class of the lecture is also much younger than the patients of a hospital. In which direction do these two facts move the code height that the display needs?

## 2. Luminance and contrast: The colour pairs of the clinic app &#91;[1](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-02/lecture.html#ref-1)&#93;

*Lecture: [Luminance and contrast](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-02/lecture.html#luminance-and-contrast)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "fits" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-02/"
               "NB02_perception_and_colour.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-2":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** The contrast ratio of the Web Content Accessibility Guidelines (WCAG) needs eleven arithmetic steps per pair of colours. An app has dozens of pairs in several themes. Written once as functions, the formulas audit a whole palette in one run. They also draw a map of all 65,536 pairs of grey text on a grey background, which no one would compute by hand.

**From the formulas to the code.** In the code, the channel value $V$ is `v`, and `linearize` returns the linear value $c$. The weights $0.2126$, $0.7152$ and $0.0722$ are the array `W_LUM`, and `relative_luminance` returns $L$. In `contrast_ratio`, `np.maximum` picks the lighter colour $L_1$ and `np.minimum` the darker one $L_2$.

In [ ]:
def linearize(v):
    """sRGB channel values from 0 to 255 -> linear light from 0 to 1."""
    c = np.asarray(v, float) / 255
    return np.where(c <= 0.04045, c / 12.92, ((c + 0.055) / 1.055) ** 2.4)

W_LUM = np.array([0.2126, 0.7152, 0.0722])

def relative_luminance(rgb):
    """L of one colour (3 values) or of many colours (one row each)."""
    return linearize(rgb) @ W_LUM

def contrast_ratio(c1, c2):
    """WCAG contrast ratio of two colours given as arrays from 0 to 255."""
    L1, L2 = relative_luminance(c1), relative_luminance(c2)
    return (np.maximum(L1, L2) + 0.05) / (np.minimum(L1, L2) + 0.05)

grey118, white, black = hex_to_rgb("#767676"), hex_to_rgb("#FFFFFF"), np.zeros(3)
print(f"L of #767676 = {relative_luminance(grey118):.4f}")
print(f"#767676 on white: {contrast_ratio(grey118, white):.2f} : 1")
print(f"black on white:   {contrast_ratio(black, white):.2f} : 1")

**Python note.** `np.where(condition, a, b)` takes `a` where the condition holds and `b` elsewhere, element by element, so `linearize` applies the two branches of the formula to every channel at once. `linearize(rgb) @ W_LUM` is the weighted sum of the three channels, for one colour or for a whole palette.

In [ ]:
REQUIRED = {"text": 4.5, "large": 3.0}     # WCAG 2.2, level AA
audit = pd.DataFrame(CLINIC_PAIRS, columns=["role", "fg", "bg", "size", "exempt"])
audit["ratio"] = [contrast_ratio(hex_to_rgb(f), hex_to_rgb(b))
                  for f, b in zip(audit["fg"], audit["bg"])]
audit["needs"] = audit["size"].map(REQUIRED)
audit["AA"] = np.where(audit["exempt"], "exempt",
                       np.where(audit["ratio"] >= audit["needs"], "pass", "fail"))
audit["AAA"] = audit["ratio"] >= audit["needs"] + np.where(
    audit["size"] == "text", 2.5, 1.5)
print(audit.round(2).to_string(index=False))
print("pairs that fail AA:", (audit["AA"] == "fail").sum(), "of",
      (~audit["exempt"]).sum())

**Python note.** `map(REQUIRED)` looks up every size class in the dictionary and returns the required ratio. Two nested `np.where` calls give three outcomes. Level AAA asks for 7 instead of 4.5 and for 4.5 instead of 3. The code therefore adds 2.5 or 1.5 to the requirement of level AA, and `~` negates a mask.

In [ ]:
# Figure: the contrast ratio of every grey text on every grey background
levels = np.arange(256)
L_grey = relative_luminance(np.repeat(levels[:, None], 3, axis=1))
L_text, L_bg = np.meshgrid(L_grey, L_grey)       # all 65,536 pairs at once
ratio_map = (np.maximum(L_text, L_bg) + 0.05) / (np.minimum(L_text, L_bg) + 0.05)
fig, ax = plt.subplots(figsize=(6.2, 5))
im = ax.imshow(np.log10(ratio_map), origin="lower", cmap="viridis",
               extent=[0, 255, 0, 255])
cs = ax.contour(levels, levels, ratio_map, levels=[3, 4.5, 7],
                colors=["#FFFFFF", "#FFD27F", "#FF8C7A"], linewidths=1.5)
ax.clabel(cs, fmt=lambda v: f"{v:g}:1")
ax.set_xlabel("grey level of the text (0 to 255)")
ax.set_ylabel("grey level of the background (0 to 255)")
ax.set_title("Contrast ratio of grey on grey")
fig.colorbar(im, label="log10 of the contrast ratio")
plt.tight_layout()
plt.show()

**Python note.** `levels[:, None]` turns the 256 levels into a column, and `np.repeat(..., 3, axis=1)` copies it three times, so each row is one grey with three equal channels. `np.meshgrid` pairs every text level with every background level. `contour` draws the lines where the ratio equals 3, 4.5 and 7, and the logarithm spreads the colours of the map more evenly.

**Exercise B.** Find the lightest grey with three equal channels whose text on white reaches the ratio 4.5 of level AA. Store its level from 0 to 255 in `lightest_grey`. The array `levels` and the function `contrast_ratio` are defined above.

In [ ]:
lightest_grey = None   # your answer

check("Exercise B", lightest_grey)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
greys = np.repeat(levels[:, None], 3, axis=1)
ok = contrast_ratio(greys, white) >= 4.5
lightest_grey = int(levels[ok].max())

check("Exercise B", lightest_grey)

**Explore.** Choose a text colour and a background colour and read the ratio and the verdict. If the controls do not appear, run the cell again.

In [ ]:
import ipywidgets as W

def explore(text="#767676", background="#FFFFFF"):
    r = contrast_ratio(hex_to_rgb(text), hex_to_rgb(background))
    print(f"contrast ratio {r:.2f} : 1")
    for need, what in [(4.5, "AA, normal text"), (3.0, "AA, large text"),
                       (7.0, "AAA, normal text"), (3.0, "non-text elements")]:
        print(f"  {what:<18} needs {need:>3} : 1 -> " + ("pass" if r >= need else "fail"))

W.interact(explore,
           text=W.ColorPicker(value="#767676", description="text"),
           background=W.ColorPicker(value="#FFFFFF", description="background"));

**Python note.** `W.ColorPicker` draws a colour field that opens the colour chooser of the browser, and its value is the colour code as text. `W.interact` calls `explore` again whenever a colour changes.

**Observe and reflect.** Many design systems use a light grey such as #999999 for secondary text. Compute its ratio on white. Why might a designer choose it anyway, and who pays for the choice?

## 3. Colour vision deficiency: The wayfinding lines &#91;[9](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-02/lecture.html#ref-9)&#93;

*Lecture: [Measuring colour difference](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-02/lecture.html#measuring-colour-difference)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "audit" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-02/"
               "NB02_perception_and_colour.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-3":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** A simulation of colour vision deficiency multiplies every colour by a matrix, and a colour difference needs a chain of four formulas. Applied to a palette under four kinds of vision, that is a few hundred steps. Python runs them in a few lines and links them to the errors that you made in the 3D corridor of Part C.

**From the formulas to the code.** The matrices $M$ of Machado, Oliveira and Fernandes are the dictionary `MACHADO`. The function `simulate` linearizes a colour, multiplies it with $M$ and encodes it again as sRGB. In `to_lab`, the matrix `XYZ` turns linear values into $X$, $Y$ and $Z$, the function `f` is $f(t)$ of the lecture, and `WHITE` holds $X_n$, $Y_n$ and $Z_n$. `delta_e` is $\Delta E^*_{ab}$.

In [ ]:
MACHADO = {                                       # severity 1.0
    "protan": np.array([[0.152286, 1.052583, -0.204868],
                        [0.114503, 0.786281, 0.099216],
                        [-0.003882, -0.048116, 1.051998]]),
    "deutan": np.array([[0.367322, 0.860646, -0.227968],
                        [0.280085, 0.672501, 0.047413],
                        [-0.011820, 0.042940, 0.968881]]),
    "tritan": np.array([[1.255528, -0.076749, -0.178779],
                        [-0.078411, 0.930809, 0.147602],
                        [0.004733, 0.691367, 0.303900]]),
}

def to_srgb(c):
    """Linear light from 0 to 1 -> sRGB channel values from 0 to 255."""
    c = np.clip(c, 0, 1)
    return 255 * np.where(c <= 0.0031308, 12.92 * c, 1.055 * c ** (1 / 2.4) - 0.055)

def simulate(rgb, kind):
    """How a colour (or a palette, one row each) looks to a dichromat."""
    if kind == "typical":
        return np.asarray(rgb, float)
    return to_srgb(linearize(rgb) @ MACHADO[kind].T)

XYZ = np.array([[0.4124, 0.3576, 0.1805],
                [0.2126, 0.7152, 0.0722],
                [0.0193, 0.1192, 0.9505]])
WHITE = XYZ.sum(axis=1)                          # X_n, Y_n, Z_n of sRGB white

def f(t):
    return np.where(t > (6 / 29) ** 3, np.cbrt(t), t / (3 * (6 / 29) ** 2) + 4 / 29)

def to_lab(rgb):
    """CIELAB values L*, a*, b* of colours given from 0 to 255."""
    fx, fy, fz = np.moveaxis(f((linearize(rgb) @ XYZ.T) / WHITE), -1, 0)
    return np.stack([116 * fy - 16, 500 * (fx - fy), 200 * (fy - fz)], axis=-1)

def delta_e(c1, c2):
    """CIE 1976 colour difference of two colours given from 0 to 255."""
    return np.linalg.norm(to_lab(c1) - to_lab(c2), axis=-1)

green, red = hex_to_rgb(STATUS["confirmed"]), hex_to_rgb(STATUS["cancelled"])
print("Lab of the status green:", to_lab(green).round(1))
print("Lab of the status red:  ", to_lab(red).round(1))
for kind in ["typical", "protan", "deutan", "tritan"]:
    g, r = simulate(green, kind), simulate(red, kind)
    print(f"{kind:<8} {rgb_to_hex(g)} vs {rgb_to_hex(r)}  delta E = {delta_e(g, r):6.1f}")

**Python note.** `linearize(rgb) @ MACHADO[kind].T` multiplies every colour of a palette with the matrix, as in the Python step. `np.moveaxis(..., -1, 0)` moves the three channels to the front, so that `fx, fy, fz = ...` unpacks them, and `np.stack(..., axis=-1)` puts the three results back together. `np.linalg.norm` is the square root of the sum of squares, the length of the difference.

In [ ]:
names_l = list(LINES)
line_rgb = np.array([hex_to_rgb(LINES[n]) for n in names_l])
pairs = [(i, j) for i in range(4) for j in range(i + 1, 4)]
table3 = pd.DataFrame(index=[f"{names_l[i]}-{names_l[j]}" for i, j in pairs])
for kind in ["typical", "protan", "deutan", "tritan"]:
    seen = simulate(line_rgb, kind)
    table3[kind] = [delta_e(seen[i], seen[j]) for i, j in pairs]
print("delta E between the four lines, as seen with each kind of vision")
print(table3.round(1))
print("closest pair:", table3.idxmin().to_dict())

**Python note.** A table can start with only an index and get its columns one by one. `idxmin()` returns, for each column, the index of its smallest value, and `to_dict()` prints the result as a dictionary.

Part C of the [interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-02/lab.html) ends with the button *Copy the results for Colab*. Paste the copied text into `MY_LINES`. Left empty, the variable makes the code use a simulated class of 24 students. Their errors and times are assumptions of this Colab notebook, which make errors more likely when the target line looks similar to another line.

In [ ]:
# Paste the results of lab Part C between the triple quotes.
MY_LINES = """
"""

CONDITIONS = {"typical": "typical", "deutan": "deutan", "deutan_symbols": "deutan"}

def nearest_delta_e(target, kind):
    """delta E between the target line and the most similar other line."""
    seen = simulate(line_rgb, kind)
    i = names_l.index(target)
    return min(delta_e(seen[i], seen[j]) for j in range(4) if j != i)

def simulated_lines_class(n_students=24, seed=RANDOM_STATE):
    """Eight trials per condition and student, with assumed errors and times."""
    rng = np.random.default_rng(seed)
    rows = []
    for student in range(1, n_students + 1):
        for cond, kind in CONDITIONS.items():
            for trial, target in enumerate(rng.permutation(names_l * 2), start=1):
                de = nearest_delta_e(target, kind)
                if cond == "deutan_symbols":
                    p_err, rt = 0.03, 1.6
                else:
                    p_err, rt = 0.02 + 0.5 * np.exp(-de / 12), 1.1 + 1.8 * np.exp(-de / 20)
                ok = int(rng.random() > p_err)
                rows.append([student, cond, trial, target, ok,
                             round(1000 * rt * rng.lognormal(0, 0.25))])
    return pd.DataFrame(rows, columns=["student", "condition", "trial", "target",
                                       "correct", "rt_ms"])

if MY_LINES.strip():
    lines = pd.read_csv(io.StringIO(MY_LINES.strip()))
    lines["student"] = 1
    source = "your own trials from the interactive lab"
else:
    lines = simulated_lines_class()
    source = "a simulated class of 24 students"
lines["nearest_dE"] = [nearest_delta_e(t, CONDITIONS[c])
                       for t, c in zip(lines["target"], lines["condition"])]
print("Data:", source, "|", len(lines), "trials")
by_cond = lines.groupby("condition").agg(
    error_rate=("correct", lambda c: 1 - c.mean()),
    mean_rt_correct_ms=("rt_ms", "mean"))
by_cond["mean_rt_correct_ms"] = lines[lines["correct"] == 1].groupby(
    "condition")["rt_ms"].mean()
print(by_cond.reindex(list(CONDITIONS)).round(3))

**Python note.** `names_l * 2` repeats the list of four names, so every colour is the target twice per condition, and `rng.permutation` shuffles the order. In `agg`, each new column is a pair of an existing column and a summary, here a small function written with `lambda`, a function without a name.

In [ ]:
# Figure: the colours as seen with four kinds of vision, and your errors
kinds = ["typical", "protan", "deutan", "tritan"]
swatch_rgb = np.vstack([line_rgb, green, red])
img = np.stack([simulate(swatch_rgb, k) for k in kinds]) / 255
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw={"width_ratios": [1.2, 1]})
ax[0].imshow(img, aspect="auto")
ax[0].set_xticks(range(6), names_l + ["status\ngreen", "status\nred"])
ax[0].set_yticks(range(4), kinds)
ax[0].grid(False)
ax[0].set_title("Four lines and two status colours")
bins = pd.cut(lines["nearest_dE"], [0, 20, 40, 60, 200])
err = 1 - lines.groupby(bins, observed=True)["correct"].mean()
ax[1].bar([str(b) for b in err.index], err.values, color="#5A4BB5")
ax[1].set_xlabel("delta E to the most similar other line")
ax[1].set_ylabel("error rate")
ax[1].set_title("Errors against colour difference")
plt.tight_layout()
plt.show()

**Python note.** `imshow` draws an array of shape (rows, columns, 3) as an image, one square per colour, with values between 0 and 1. `pd.cut` sorts each trial into a band of $\Delta E$, and `observed=True` keeps only the bands that occur.

**Exercise C.** A redesign replaces the red of the status dots by an amber, #C98510, and darkens the green to #2E8B57. Store the colour difference $\Delta E^*_{ab}$ of the two colours, as seen with deuteranopia, in `de_green_amber`.

In [ ]:
de_green_amber = None   # your answer

check("Exercise C", de_green_amber)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
de_green_amber = delta_e(simulate(hex_to_rgb("#2E8B57"), "deutan"),
                         simulate(hex_to_rgb("#C98510"), "deutan"))

check("Exercise C", de_green_amber)

**Observe and reflect.** Symbols lowered the errors with simulated deuteranopia, but did they make the trials slower than the typical colours? What does redundant coding cost a user with typical vision, and is the cost worth it?

## 4. Colour and emotion: Your ratings against a published model &#91;[11](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-02/lecture.html#ref-11)&#93;

*Lecture: [The psychological effects of colour](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-02/lecture.html#the-psychological-effects-of-colour)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "lines" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-02/"
               "NB02_perception_and_colour.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-4":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Do your own feelings about colours follow the published model? Sixteen colours, two ratings each and two properties of every colour make a small regression. Python fits it in one line for you and for a whole class, and compares the weights with those of Valdez and Mehrabian.

**From the formulas to the code.** In the code, the brightness $B$ and the saturation $S$ in standard units are `zB` and `zS`. The weights of the published equations are the dictionary `VM`. The function `standardize` subtracts the mean and divides by the standard deviation, and `np.linalg.lstsq` finds the weights that fit the ratings best.

In [ ]:
VM = {"pleasure": (0.69, 0.22), "arousal": (-0.31, 0.60), "dominance": (-0.76, 0.32)}

def predict_pad(zB, zS):
    """Standardized pleasure, arousal and dominance of Valdez and Mehrabian."""
    return {k: wB * zB + wS * zS for k, (wB, wS) in VM.items()}

def standardize(x):
    x = np.asarray(x, float)
    return (x - x.mean()) / x.std()

for zB, zS in [(1.0, -0.5), (-0.6, 1.4)]:
    pad = predict_pad(zB, zS)
    print(f"B = {zB:+.1f}, S = {zS:+.1f}: " +
          ", ".join(f"{k} {v:+.2f}" for k, v in pad.items()))

**Python note.** `for k, (wB, wS) in VM.items()` unpacks each pair of weights directly in the loop. A dictionary comprehension builds the three predictions at once, and the format `+.2f` always prints the sign.

Part D of the [interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-02/lab.html) ends with the button *Copy the results for Colab*. Paste the copied text into `MY_ROOM`. The code always simulates a class of 24 students as well. Their ratings are assumptions of this Colab notebook, built around the published weights with differences between students, so they agree with the model by construction. Your own ratings are the real test, and they appear as student 0.

In [ ]:
# Paste the results of lab Part D between the triple quotes.
MY_ROOM = """
"""

HUES = [0, 55, 135, 215]                         # red, yellow, green, blue
ROOM = pd.DataFrame([(h, s, b) for h in HUES for b in (0.9, 0.5) for s in (0.3, 0.8)],
                    columns=["hue", "sat", "bri"])
ROOM["hex"] = [rgb_to_hex(255 * np.array(colorsys.hsv_to_rgb(h / 360, s, b)))
               for h, s, b in ROOM[["hue", "sat", "bri"]].values]

def simulated_room_class(n_students=24, seed=RANDOM_STATE):
    """Ratings from 1 to 9 for the 16 colours, with assumed personal weights."""
    rng = np.random.default_rng(seed)
    zB, zS = standardize(ROOM["bri"]), standardize(ROOM["sat"])
    rows = []
    for student in range(1, n_students + 1):
        w = {k: rng.normal(VM[k], 0.2) for k in ("pleasure", "arousal")}
        for i, c in ROOM.iterrows():
            pl = 5 + 1.5 * (w["pleasure"][0] * zB[i] + w["pleasure"][1] * zS[i])
            ar = 5 + 1.5 * (w["arousal"][0] * zB[i] + w["arousal"][1] * zS[i])
            pl, ar = np.clip(np.round(np.array([pl, ar]) + rng.normal(0, 1, 2)), 1, 9)
            rows.append([student, c["hex"], c["hue"], c["sat"], c["bri"], pl, ar])
    return pd.DataFrame(rows, columns=["student", "hex", "hue", "sat", "bri",
                                       "pleasure", "arousal"])

room = simulated_room_class()
if MY_ROOM.strip():
    mine = pd.read_csv(io.StringIO(MY_ROOM.strip()))
    mine = mine[mine["trial"] > 0].assign(student=0)
    room = pd.concat([mine[room.columns], room])
print("students:", room["student"].nunique(), "| ratings:", len(room))
room.head()

**Python note.** `colorsys.hsv_to_rgb` from the standard library of Python turns hue, saturation and brightness, each from 0 to 1, into red, green and blue from 0 to 1. `assign(student=0)` adds a column with the same value in every row, and `room.columns` selects the columns in the order of the simulated table.

In [ ]:
def standardized_weights(d, rating):
    """Weights of zB and zS for one student, all variables in standard units."""
    X = np.column_stack([standardize(d["bri"]), standardize(d["sat"])])
    y = standardize(d[rating])
    w, *_ = np.linalg.lstsq(X, y, rcond=None)
    return w

rows = []
for student, d in room.groupby("student"):
    if d["pleasure"].std() > 0 and d["arousal"].std() > 0:
        rows.append([student, *standardized_weights(d, "pleasure"),
                     *standardized_weights(d, "arousal")])
betas = pd.DataFrame(rows, columns=["student", "P_B", "P_S", "A_B", "A_S"])
betas = betas.set_index("student")
compare = pd.DataFrame({"Valdez and Mehrabian": [0.69, 0.22, -0.31, 0.60],
                        "class mean": betas.drop(index=0, errors="ignore").mean()
                        .values}, index=betas.columns)
if 0 in betas.index:
    compare["you"] = betas.loc[0].values
print(compare.round(2))

**Python note.** `np.column_stack` puts two arrays side by side as the columns of a table. `np.linalg.lstsq(X, y)` solves the least-squares problem, and `w, *_ = ...` keeps the first result and drops the rest. The star in `[student, *weights]` unpacks the two weights into the list. A student who gave every colour the same rating has no spread, so the condition `std() > 0` skips that student.

In [ ]:
# Figure: the weights side by side, and the pleasure model as a surface in 3D
fig = plt.figure(figsize=(11, 4))
ax0 = fig.add_subplot(1, 2, 1)
xpos = np.arange(4)
width = 0.8 / len(compare.columns)
for k, col in enumerate(compare.columns):
    ax0.bar(xpos + k * width, compare[col], width, label=col,
            color=["#8A93A3", "#C9C3E6", "#5A4BB5"][k])
ax0.set_xticks(xpos + width, ["pleasure\nbrightness", "pleasure\nsaturation",
                              "arousal\nbrightness", "arousal\nsaturation"])
ax0.axhline(0, color="#1E2433", lw=0.8)
ax0.set_ylabel("standardized weight")
ax0.set_title("Weights of brightness and saturation")
ax0.legend(fontsize=8)
ax1 = fig.add_subplot(1, 2, 2, projection="3d")
gB, gS = np.meshgrid(np.linspace(-1.5, 1.5, 25), np.linspace(-1.5, 1.5, 25))
wPB, wPS = compare["class mean"].iloc[0], compare["class mean"].iloc[1]
ax1.plot_surface(gB, gS, wPB * gB + wPS * gS, cmap="viridis", alpha=0.75)
ax1.set_xticks([-1, 0, 1])
ax1.set_yticks([-1, 0, 1])
ax1.set_xlabel("brightness B")
ax1.set_ylabel("saturation S")
ax1.set_zlabel("pleasure")
ax1.set_title("Pleasure predicted from the class weights")
plt.tight_layout()
plt.show()

**Python note.** `fig.add_subplot(1, 2, 2, projection="3d")` adds a panel with three axes, and `plot_surface` draws the predicted pleasure over a grid of brightness and saturation. A plane appears, because the model adds the two weighted properties.

**Exercise D.** A colour has the brightness $B = 0.5$ and the saturation $S = 1.2$ in standard units. Store the arousal that the equations of Valdez and Mehrabian predict in `arousal_pred`.

In [ ]:
arousal_pred = None   # your answer

check("Exercise D", arousal_pred)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
arousal_pred = predict_pad(0.5, 1.2)["arousal"]

check("Exercise D", arousal_pred)

**Observe and reflect.** Compare your own weights with the published ones. Where do they differ most, and what else besides brightness and saturation might have shaped your ratings of a room?

## 5. The clinic app, stage 2: An audit of colour, contrast and size

*Lecture: [The clinic app, stage 2: An audit of colour, contrast and size](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-02/lecture.html#the-clinic-app-stage-2)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "betas" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-02/"
               "NB02_perception_and_colour.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-5":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** The audit joins all the functions of this Colab notebook: Contrast, visual angle, simulated vision, colour difference and the colour-emotion model. Run together, they turn the screens of the app into one table of findings. Week 10 repeats the audit after the redesign.

In [ ]:
AUDIT = {
    "pairs failing AA": int((audit["AA"] == "fail").sum()),
    "pairs audited": int((~audit["exempt"]).sum()),
    "texts below 0.2 deg": texts.loc[~texts["fluent"], "text"].tolist(),
    "kiosk prompt (deg)": round(float(texts.loc[2, "angle_deg"]), 3),
}
for kind in ["typical", "protan", "deutan", "tritan"]:
    AUDIT[f"status dE, {kind}"] = round(float(
        delta_e(simulate(green, kind), simulate(red, kind))), 1)

def ratio_with_glare(c1, c2, r=0.2):
    """Contrast ratio when reflected light adds r to both luminances."""
    L1, L2 = relative_luminance(c1), relative_luminance(c2)
    return (max(L1, L2) + 0.05 + r) / (min(L1, L2) + 0.05 + r)

audit["ratio_sun"] = [ratio_with_glare(hex_to_rgb(f), hex_to_rgb(b))
                      for f, b in zip(audit["fg"], audit["bg"])]
AUDIT["pairs passing AA in the sun"] = int(
    ((audit["ratio_sun"] >= audit["needs"]) & ~audit["exempt"]).sum())

WALLS = {"current orange": "#D35400", "pale blue": "#A9CCE3",
         "light sand": "#E8D8B0"}
mB, sB = ROOM["bri"].mean(), ROOM["bri"].std(ddof=0)
mS, sS = ROOM["sat"].mean(), ROOM["sat"].std(ddof=0)
for name, code in WALLS.items():
    h, s, b = colorsys.rgb_to_hsv(*(hex_to_rgb(code) / 255))
    pad = predict_pad((b - mB) / sB, (s - mS) / sS)
    AUDIT[f"wall {name}"] = {k: round(float(v), 2) for k, v in pad.items()}
for k, v in AUDIT.items():
    print(f"{k:<28} {v}")

**Python note.** `texts.loc[2, "angle_deg"]` reads one cell by its row and column, and `tolist()` turns a column into a plain list. `colorsys.rgb_to_hsv` is the inverse of `hsv_to_rgb`, and `std(ddof=0)` divides by $n$, as `standardize` does, so the walls use the same standard units as the sixteen colours of Part D.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12.5, 3.9))
ok = audit[~audit["exempt"]]
cols = np.where(ok["ratio"] >= ok["needs"], "#2E8B57", "#B53F2E")
ax[0].barh(ok["role"], ok["ratio"], color=cols)
ax[0].axvline(4.5, color="#1E2433", ls="--", lw=1)
ax[0].set_xlabel("contrast ratio (dashed: 4.5)")
ax[0].invert_yaxis()
ax[0].set_title("Colour pairs")
ax[0].tick_params(axis="y", labelsize=7.5)
cols = np.where(texts["fluent"], "#2E8B57", "#B53F2E")
ax[1].barh(texts["text"], texts["angle_deg"], color=cols)
ax[1].axvline(CRITICAL_DEG, color="#1E2433", ls="--", lw=1)
ax[1].set_xlabel("visual angle (deg; dashed: 0.2)")
ax[1].invert_yaxis()
ax[1].set_title("Text sizes")
ax[1].tick_params(axis="y", labelsize=7.5)
de = [AUDIT[f"status dE, {k}"] for k in kinds]
ax[2].bar(kinds, de, color=["#5A4BB5", "#C98510", "#C98510", "#C98510"])
ax[2].set_ylabel("delta E of green and red")
ax[2].set_title("Status colours")
plt.tight_layout()
plt.show()

**Python note.** `np.where` chooses a colour for every bar, green for a pass and red for a fail. `barh` draws horizontal bars with the role as the label, which suits long names.

**Exercise E.** Store the number of colour pairs of the clinic app that fail level AA, without the exempt pair of the disabled button, in `n_fail`. Count it from the table `audit` instead of copying it.

In [ ]:
n_fail = None   # your answer

check("Exercise E", n_fail)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
n_fail = int((audit["AA"] == "fail").sum())

check("Exercise E", n_fail)

**Observe and reflect.** Which finding of the audit would you fix first, and why? Consider how many patients it affects, how severe the effect is and how cheap the fix is.

## 6. Your application: The application switch

*Lecture: [Your application](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-02/lecture.html#your-application)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "AUDIT" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-02/"
               "NB02_perception_and_colour.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-6":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Each app below is just a list of colour pairs and texts. The functions of Sections 1 and 2 audit any of them after one change of a name, which is how design tools check a whole design system.

The colour pairs and the texts below are illustrative, as in the lecture. Set `APP` to one of the five names and run the two cells.

In [ ]:
APP = "banking"
# choose: clinic, banking, e_government, food_delivery, course_registration

APPS = {
    "clinic": dict(pairs=[(r, f, b, s) for r, f, b, s, e in CLINIC_PAIRS if not e],
                   texts=CLINIC_TEXTS),
    "banking": dict(pairs=[("Balance", "#0B3D2E", "#FFFFFF", "large"),
                           ("Transaction dates", "#9AA5A0", "#FFFFFF", "text"),
                           ("Button label", "#FFFFFF", "#00A86B", "text"),
                           ("Warning", "#F39C12", "#FFFFFF", "text"),
                           ("Footer links", "#7F8C8D", "#F4F6F6", "text")],
                    texts=[("Phone, amounts", 3.0, 350), ("Phone, fine print", 1.2, 350)]),
    "e_government": dict(pairs=[("Body text", "#333333", "#FFFFFF", "text"),
                                ("Form hints", "#999999", "#FFFFFF", "text"),
                                ("Header", "#FFFFFF", "#C8102E", "large"),
                                ("Step indicator", "#BBBBBB", "#FFFFFF", "text")],
                         texts=[("Laptop, body text", 2.4, 550),
                                ("Laptop, form hints", 1.6, 550)]),
    "food_delivery": dict(pairs=[("Dish names", "#222222", "#FFFFFF", "text"),
                                 ("Prices", "#FF5A5F", "#FFFFFF", "text"),
                                 ("Delivery time", "#FFFFFF", "#FFB400", "text"),
                                 ("Promotion", "#FFFFFF", "#FF5A5F", "large")],
                          texts=[("Phone, dish names", 2.3, 350),
                                 ("Phone, allergy notes", 1.3, 350)]),
    "course_registration": dict(pairs=[("Course codes", "#1A1A1A", "#FFFFFF", "text"),
                                       ("Credits", "#8E8E8E", "#FFFFFF", "text"),
                                       ("Clash warning", "#FFFFFF", "#E67E22", "text"),
                                       ("Selected row", "#1A1A1A", "#D6EAF8", "text")],
                                texts=[("Laptop, timetable", 1.8, 550),
                                       ("Wall screen, room numbers", 25, 8000)]),
}
app = APPS[APP]
pairs_app = pd.DataFrame(app["pairs"], columns=["role", "fg", "bg", "size"])
pairs_app["ratio"] = [contrast_ratio(hex_to_rgb(f), hex_to_rgb(b))
                      for f, b in zip(pairs_app["fg"], pairs_app["bg"])]
pairs_app["pass_AA"] = pairs_app["ratio"] >= pairs_app["size"].map(REQUIRED)
texts_app = pd.DataFrame(app["texts"], columns=["text", "h_mm", "d_mm"])
texts_app["angle_deg"] = visual_angle(texts_app["h_mm"], texts_app["d_mm"])
print(pairs_app.round(2).to_string(index=False))
print(texts_app.round(3).to_string(index=False))

**Python note.** The list comprehension with a condition, `[... for ... if not e]`, keeps only the pairs of the clinic app that WCAG does not exempt. Every app is a dictionary with the same two keys, so the code below it works for all of them.

In [ ]:
worst = pairs_app.loc[pairs_app["ratio"].idxmin()]
print(f"{APP}: {(~pairs_app['pass_AA']).sum()} of {len(pairs_app)} pairs fail AA")
print(f"weakest pair: {worst['role']} ({worst['fg']} on {worst['bg']}), "
      f"{worst['ratio']:.2f} : 1")
fig, ax = plt.subplots(figsize=(7.5, 0.55 * len(pairs_app) + 0.6))
for i, row in pairs_app.iterrows():
    ax.add_patch(plt.Rectangle((0, -i - 0.9), 3.2, 0.8, color=row["bg"], ec="#999999"))
    ax.text(1.6, -i - 0.5, row["role"], color=row["fg"], ha="center", va="center",
            fontsize=12, fontweight="bold" if row["size"] == "large" else "normal")
    ax.text(3.4, -i - 0.5, f"{row['ratio']:.2f} : 1  " + ("pass" if row["pass_AA"]
            else "fail"), va="center", fontsize=10)
ax.set_xlim(0, 5)
ax.set_ylim(-len(pairs_app), 0)
ax.axis("off")
ax.set_title(f"The colour pairs of {APP}")
plt.tight_layout()
plt.show()

**Python note.** `plt.Rectangle` draws a box in the colour of the background, and `ax.text` writes the role in the colour of the text. The figure thus shows each pair as it looks, and `ax.axis("off")` hides the axes.

**Observe and reflect.** Compare the weakest pairs of two apps. Is the problem a light grey, a saturated colour or white text on a bright colour? Which fix keeps the brand colour of the app?

## Application challenges

Each challenge transfers the ideas of the week to a field of computing. Choose the one closest to your interests and solve it in a copy of this Colab notebook.

| Field | Challenge |
|---|---|
| Software engineering | Write unit tests for `contrast_ratio`: Black on white must give 21, any colour on itself 1, and swapping the two colours must not change the result. |
| Web development | Write a function that takes a text colour and a background and returns the closest darker text colour, with the same hue, that passes level AA. |
| Mobile development | Phone screens are read at about 35 cm and tablets at about 45 cm. Compute the x-height in millimetres that reaches 0.3 degrees on each device. |
| Game development | A game marks friends in green and enemies in red. Simulate the two colours for the three deficiencies and propose a pair, or a second cue, that stays apart in all of them. |
| Embedded systems and the Internet of Things | A charger shows its state with one LED that turns from red to green. Compute $\Delta E^*_{ab}$ of the two states with deuteranopia and design a signal that does not depend on colour. |
| Data science | A chart uses the default colour cycle of Matplotlib. Simulate its first six colours with deuteranopia and find the pair with the smallest $\Delta E^*_{ab}$. |
| Cybersecurity | A password field shows strength with a bar from red to green. Redesign the signal so that a user with protanopia can read it, and check your colours with the functions of Section 3. |
| Health informatics | Patient monitors use red for alarms. Audit the contrast of red alarm text on the dark background #101820 and propose a version that passes level AAA. |

## After the Colab notebook

The self-assessment and the reflection are in the [interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-02/lab.html), tabs *Check yourself* and *Reflect and export*. The weekly task and the research assignment are on the [week overview](https://github.com/utkukose/human-computer-interaction-NB-lecture/tree/main/weeks/week-02).

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

This course is updated in line with current developments in the field. First version: November 2026.

</div>